# Wisata Training: Decision-Driven Data Preparation dan Audit

Notebook ini menjadi jalur utama untuk data wisata MuterBandung. Polanya dibuat seperti notebook penginapan: setiap tahap kecil menjalankan pengecekan, melihat output, lalu mencatat keputusan singkat.

Catatan: notebook lama disimpan sebagai arsip. Tahap awal ini belum masuk ke LLM atau perubahan model.


## Tahap 0 - Menetapkan Environment dan Artefak

Tujuan: memastikan notebook membaca folder wisata yang benar dan memakai dataset curated terbaru sebagai basis audit.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 90)

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "Wisata_Workspace").exists():
            return candidate
    raise FileNotFoundError("Wisata_Workspace tidak ditemukan dari cwd saat ini.")

PROJECT_ROOT = find_project_root()
WISATA_DIR = PROJECT_ROOT / "Wisata_Workspace"
CURATED_DIR = WISATA_DIR / "01_Dataset" / "3_Curated"
DOC_DIR = WISATA_DIR / "03_Dokumentasi"

paths = {
    "dataset_utama": CURATED_DIR / "DATABASE_WISATA_LABELED_V2_REVIEWED.csv",
    "validation_json": CURATED_DIR / "data_validation_results.json",
    "validation_report": DOC_DIR / "DATA_VALIDATION_REPORT_2026-05-25.md",
    "sentiment_score": WISATA_DIR / "01_Dataset" / "SENTIMENT_SCORES_PER_LOKASI.csv",
    "manual_batch3": CURATED_DIR / "manual_review_batch3_remaining_46_after_status_facility_2026-05-27.csv",
}

artifact_status = []
for name, path in paths.items():
    artifact_status.append({
        "artifact": name,
        "relative_path": path.relative_to(PROJECT_ROOT).as_posix(),
        "exists": path.exists(),
        "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else None,
        "modified_at": path.stat().st_mtime if path.exists() else None,
    })

artifact_status_df = pd.DataFrame(artifact_status)
display(artifact_status_df.drop(columns=["modified_at"]))
print("PROJECT_ROOT:", PROJECT_ROOT)


,artifact,relative_path,exists,size_kb
0,dataset_utama,Wisata_Workspace/01_Dataset/3_Curated/DATABASE_WISATA_LABELED_V2_REVIEWED.csv,True,502.21
1,validation_json,Wisata_Workspace/01_Dataset/3_Curated/data_validation_results.json,True,13.01
2,validation_report,Wisata_Workspace/03_Dokumentasi/DATA_VALIDATION_REPORT_2026-05-25.md,True,6.43
3,sentiment_score,Wisata_Workspace/01_Dataset/SENTIMENT_SCORES_PER_LOKASI.csv,True,27.45
4,manual_batch3,Wisata_Workspace/01_Dataset/3_Curated/manual_review_batch3_remaining_46_after_status_f...,True,21.12


PROJECT_ROOT: D:\File\file\Fauzan Lubada\PIJAK


### Keputusan: environment wisata layak dipakai

Semua artefak awal yang dibutuhkan tersedia. Untuk tahap ini, dataset utama yang dipakai adalah `DATABASE_WISATA_LABELED_V2_REVIEWED.csv`, bukan notebook lama atau file Colab eksperimen.


## Fase A - Lineage Dataset Utama

Tujuan: melihat jumlah data, jumlah kolom, ID unik, dan status snapshot dataset yang akan dipakai untuk audit berikutnya.


In [2]:
df = pd.read_csv(paths["dataset_utama"], dtype=str, keep_default_na=False)
validation = json.loads(paths["validation_json"].read_text(encoding="utf-8"))
summary = validation.get("summary", {})

lineage_summary = pd.DataFrame([
    {"metric": "rows_csv", "value": len(df)},
    {"metric": "columns_csv", "value": len(df.columns)},
    {"metric": "unique_location_id_csv", "value": df["location_id"].nunique() if "location_id" in df.columns else None},
    {"metric": "rows_validation_json", "value": summary.get("row_count")},
    {"metric": "columns_validation_json", "value": summary.get("column_count")},
    {"metric": "unique_location_id_validation", "value": summary.get("unique_location_id_count")},
    {"metric": "gate_status", "value": summary.get("gate_status")},
    {"metric": "validated_at", "value": summary.get("validated_at")},
])

display(lineage_summary)

id_check = pd.DataFrame([
    {"check": "location_id kosong", "count": int((df.get("location_id", pd.Series(dtype=str)) == "").sum())},
    {"check": "location_id duplikat", "count": int(df["location_id"].duplicated().sum()) if "location_id" in df.columns else None},
    {"check": "location_name kosong", "count": int((df.get("location_name", pd.Series(dtype=str)) == "").sum())},
])
display(id_check)


,metric,value
0,rows_csv,234
1,columns_csv,87
2,unique_location_id_csv,234
3,rows_validation_json,234
4,columns_validation_json,87
5,unique_location_id_validation,234
6,gate_status,PASS_WITH_WARNINGS
7,validated_at,2026-06-02T03:47:41.925336Z


,check,count
0,location_id kosong,0
1,location_id duplikat,0
2,location_name kosong,0


### Keputusan: snapshot wisata diterima sebagai basis audit

Jumlah baris, kolom, dan ID unik konsisten dengan hasil validation JSON. Tidak ada indikasi awal bahwa snapshot ini salah file, jadi tahap berikutnya boleh memakai dataset ini sebagai basis audit wisata.


## Fase B - Status Dataset dan Kolom Inti

Tujuan: memastikan status destinasi dan kolom penting untuk rekomendasi wisata tersedia sebelum masuk ke audit warning.


In [3]:
status_counts = df["display_status"].value_counts(dropna=False).rename_axis("display_status").reset_index(name="count") if "display_status" in df.columns else pd.DataFrame()
display(status_counts)

core_columns = [
    "location_id", "location_name", "category", "final_primary_intent", "final_core_labels",
    "display_status", "is_active_verified", "latitude", "longitude", "coordinate_verified",
    "price_min", "price_max", "price_type", "jam_buka_weekday", "jam_tutup_weekday",
    "avg_rating", "total_ulasan", "sentiment_score", "sentimen_label_lokasi",
    "sentiment_available", "media_available", "media_image_url", "parking_verified",
    "toilet_verified", "mushola_verified", "child_friendly_verified", "night_verified",
]

core_audit = []
for col in core_columns:
    exists = col in df.columns
    if exists:
        empty_count = int((df[col].astype(str).str.strip() == "").sum())
        non_empty_count = int(len(df) - empty_count)
    else:
        empty_count = None
        non_empty_count = None
    core_audit.append({
        "column": col,
        "exists": exists,
        "non_empty_count": non_empty_count,
        "empty_count": empty_count,
    })

core_audit_df = pd.DataFrame(core_audit)
display(core_audit_df)


,display_status,count
0,active_candidate,209
1,exclude_scope,19
2,temporarily_hidden,3
3,status_uncertain,3


,column,exists,non_empty_count,empty_count
0,location_id,True,234,0
1,location_name,True,234,0
2,category,True,234,0
3,final_primary_intent,True,234,0
4,final_core_labels,True,234,0
5,display_status,True,234,0
6,is_active_verified,True,87,147
7,latitude,True,234,0
8,longitude,True,234,0
9,coordinate_verified,True,234,0


### Keputusan: kolom inti cukup untuk audit rekomendasi

Kolom utama untuk status, koordinat, harga, jam buka, rating, sentiment, media, dan fasilitas sudah tersedia. Karena masih ada nilai kosong pada beberapa kolom, tahap berikutnya tetap perlu membaca warning validation, bukan langsung menganggap data final bersih.


## Fase C - Warning Validation yang Masih Aktif

Tujuan: membaca hasil validation otomatis dan menentukan bagian mana yang harus dibawa ke audit lanjutan.


In [4]:
issues = pd.DataFrame(validation.get("issues", []))
if not issues.empty:
    issue_cols = ["severity", "code", "field", "count", "message"]
    issues_view = issues[issue_cols].sort_values(["severity", "count"], ascending=[True, False]).reset_index(drop=True)
    display(issues_view)
else:
    issues_view = pd.DataFrame(columns=["severity", "code", "field", "count", "message"])
    display(issues_view)

sample_rows = []
for issue in validation.get("issues", []):
    for row in issue.get("sample_rows", [])[:5]:
        sample_rows.append({
            "code": issue.get("code"),
            "field": issue.get("field"),
            "location_id": row.get("location_id"),
            "location_name": row.get("location_name"),
        })

sample_df = pd.DataFrame(sample_rows)
display(sample_df.head(30))

summary_view = pd.DataFrame([
    {"metric": "gate_status", "value": summary.get("gate_status")},
    {"metric": "error_count", "value": summary.get("error_count")},
    {"metric": "warning_count", "value": summary.get("warning_count")},
    {"metric": "info_count", "value": summary.get("info_count")},
    {"metric": "active_candidate_count", "value": summary.get("active_candidate_count")},
])
display(summary_view)


,severity,code,field,count,message
0,INFO,I_ACTIVE_WEBSITE_MISSING,media_website,139,Active candidates without official/reference website.
1,WARNING,W_ACTIVE_VERIFICATION_MISSING,is_active_verified,147,Active candidates missing is_active_verified must not be described as verified.
2,WARNING,W_ACTIVE_FACILITY_VERIFICATION_SPARSE,facility_verified_flags,146,Active candidates with no verified facility flags require conservative LLM wording.
3,WARNING,W_ACTIVE_MEDIA_UNAVAILABLE,media_available,39,Active candidates without media need conservative UI/LLM presentation.
4,WARNING,W_ACTIVE_RATING_MISSING,avg_rating,16,Active candidates with missing avg_rating rely on runtime median fallback.
5,WARNING,W_ACTIVE_SENTIMENT_UNAVAILABLE,sentiment_available,16,Active candidates with unavailable sentiment need neutral/caveated LLM wording.
6,WARNING,W_ACTIVE_COORDINATE_UNVERIFIED,coordinate_verified,9,Active candidates with coordinate_verified=False should be treated carefully for dista...
7,WARNING,W_ZERO_PRICE_NOT_MARKED_FREE,"price_type,price_max",1,Rows with zero max price should usually be marked Gratis.
8,WARNING,W_OPEN_24H_FLAG_HOURS_MISMATCH,open_24h_verified,1,open_24h_verified=True should align with 00:00-23:59 weekday/weekend hours.


,code,field,location_id,location_name
0,W_ACTIVE_RATING_MISSING,avg_rating,LOC-029,Dusun Bambu
1,W_ACTIVE_RATING_MISSING,avg_rating,LOC-068,Pasar Cimindi
2,W_ACTIVE_RATING_MISSING,avg_rating,LOC-133,Punclut Bandung (Puncak Ciumbuleuit)
3,W_ACTIVE_RATING_MISSING,avg_rating,LOC-134,Bukit Bintang Bandung (Patahan Lembang)
4,W_ACTIVE_RATING_MISSING,avg_rating,LOC-152,Kebun Teh Rancabali
5,W_ZERO_PRICE_NOT_MARKED_FREE,"price_type,price_max",LOC-233,Kampung Wisata Pangjugjugan
6,W_OPEN_24H_FLAG_HOURS_MISMATCH,open_24h_verified,LOC-134,Bukit Bintang Bandung (Patahan Lembang)
7,W_ACTIVE_VERIFICATION_MISSING,is_active_verified,LOC-001,23 Paskal Shopping Center
8,W_ACTIVE_VERIFICATION_MISSING,is_active_verified,LOC-002,Alam Wisata Cimahi
9,W_ACTIVE_VERIFICATION_MISSING,is_active_verified,LOC-003,Alun-Alun Kota Bandung


,metric,value
0,gate_status,PASS_WITH_WARNINGS
1,error_count,0
2,warning_count,375
3,info_count,139
4,active_candidate_count,209


### Keputusan: dataset bisa lanjut, tetapi masih perlu caveat

Gate validation berada di `PASS_WITH_WARNINGS`. Tidak ada error aktif, tetapi warning fasilitas, status aktif, media, rating, sentiment, dan koordinat perlu dibawa ke tahap audit berikutnya. Untuk LLM dan rekomendasi, sistem belum boleh membuat klaim yang terlalu pasti pada field yang belum verified.


## Fase D - Scope Destinasi dan Status Tayang

Tujuan: memisahkan destinasi yang layak tampil, ditahan, tidak masuk scope, atau masih perlu keputusan. Ini penting sebelum audit teknis, karena rekomendasi hanya boleh fokus ke data yang statusnya jelas.


In [5]:
status_scope = df.groupby(["display_status"], dropna=False).agg(
    total=("location_id", "count"),
    active_verified=("is_active_verified", lambda s: int((s.astype(str).str.lower() == "true").sum())),
    active_not_verified=("is_active_verified", lambda s: int((s.astype(str).str.lower() != "true").sum())),
).reset_index().sort_values("total", ascending=False)
display(status_scope)

status_detail_cols = ["location_id", "location_name", "category", "display_status", "curation_action", "is_active_verified", "curation_note"]
status_detail_cols = [c for c in status_detail_cols if c in df.columns]
status_detail = df.loc[df["display_status"].ne("active_candidate"), status_detail_cols].copy()
display(status_detail.sort_values(["display_status", "location_name"]).head(30))

active_df = df[df["display_status"].eq("active_candidate")].copy()
active_verification_summary = pd.DataFrame([
    {"metric": "active_candidate", "count": len(active_df)},
    {"metric": "active_verified_true", "count": int((active_df["is_active_verified"].astype(str).str.lower() == "true").sum())},
    {"metric": "active_verified_not_true", "count": int((active_df["is_active_verified"].astype(str).str.lower() != "true").sum())},
])
display(active_verification_summary)


,display_status,total,active_verified,active_not_verified
0,active_candidate,209,62,147
1,exclude_scope,19,0,19
2,status_uncertain,3,0,3
3,temporarily_hidden,3,0,3


,location_id,location_name,category,display_status,curation_action,is_active_verified,curation_note
3,LOC-004,Alun-Alun Sumedang,Taman Kota,exclude_scope,remove,False,QA override: Sumedang destination is outside strict Bandung Raya scope.
8,LOC-009,Bendungan Jatigede,Wisata Alam,exclude_scope,remove,False,QA override: Jatigede/Sumedang destination is outside strict Bandung Raya scope.
14,LOC-015,Cadas Pangeran Sumedang,Tempat Sejarah,exclude_scope,remove,False,QA override: Sumedang historical site is outside strict Bandung Raya scope.
229,LOC-230,Desa Wisata Buricak Burinong,Desa Wisata,exclude_scope,remove,False,"Wisata valid, tetapi berada di Darmaraja, Sumedang/Waduk Jatigede, bukan Bandung Raya...."
36,LOC-037,Gn. Tampomas,Rekreasi Keluarga,exclude_scope,remove,False,QA override: Tampomas/Sumedang destination is outside strict Bandung Raya scope.
47,LOC-048,Kampung Karuhun Eco Green Park Sumedang,Rekreasi Keluarga,exclude_scope,remove,False,QA override: Sumedang destination is outside strict Bandung Raya scope.
48,LOC-049,Kampung Toga Villa & Resto,Tempat Kuliner,exclude_scope,remove,False,QA override: Sumedang/Toga area destination is outside strict Bandung Raya scope.
232,LOC-233,Kampung Wisata Pangjugjugan,Rekreasi Alam,exclude_scope,remove,False,QA override: Sumedang destination is outside strict Bandung Raya scope.
56,LOC-057,Masjid Agung Sumedang,Tempat Ibadah,exclude_scope,remove,False,QA override: Sumedang destination is outside strict Bandung Raya scope.
58,LOC-059,Mata Air Cikandung,Wisata Alam,exclude_scope,remove,False,QA override: Sumedang destination is outside strict Bandung Raya scope.


,metric,count
0,active_candidate,209
1,active_verified_true,62
2,active_verified_not_true,147


### Keputusan: scope rekomendasi fokus ke active candidate

Destinasi dengan `active_candidate` menjadi basis utama rekomendasi. Baris `exclude_scope`, `temporarily_hidden`, dan `status_uncertain` tidak dibuang, tetapi tidak diperlakukan setara dengan destinasi aktif sampai ada keputusan yang lebih kuat.


## Fase E - Koordinat dan Distance Readiness

Tujuan: mengecek apakah koordinat cukup aman untuk klaim jarak, radius, dan fitur ?dekat saya?. Validasi ini hanya filter kasar, bukan batas administratif presisi.


In [6]:
coord_cols = ["location_id", "location_name", "display_status", "latitude", "longitude", "coordinate_verified", "distance_from_alun_alun_km", "coordinate_audit_reason"]
coord_cols = [c for c in coord_cols if c in df.columns]
coord_df = df[coord_cols].copy()

coord_df["lat_num"] = pd.to_numeric(coord_df.get("latitude", ""), errors="coerce")
coord_df["lon_num"] = pd.to_numeric(coord_df.get("longitude", ""), errors="coerce")
coord_df["distance_num"] = pd.to_numeric(coord_df.get("distance_from_alun_alun_km", ""), errors="coerce")
coord_df["coord_numeric_ok"] = coord_df["lat_num"].notna() & coord_df["lon_num"].notna()
coord_df["inside_rough_bandung_raya_box"] = coord_df["lat_num"].between(-7.35, -6.60) & coord_df["lon_num"].between(107.20, 108.15)
coord_df["coordinate_verified_bool"] = coord_df.get("coordinate_verified", "").astype(str).str.lower().eq("true")

coord_summary = pd.DataFrame([
    {"metric": "total_rows", "count": len(coord_df)},
    {"metric": "numeric_coordinate_ok", "count": int(coord_df["coord_numeric_ok"].sum())},
    {"metric": "missing_or_invalid_coordinate", "count": int((~coord_df["coord_numeric_ok"]).sum())},
    {"metric": "inside_rough_bandung_raya_box", "count": int(coord_df["inside_rough_bandung_raya_box"].sum())},
    {"metric": "outside_rough_bandung_raya_box", "count": int((coord_df["coord_numeric_ok"] & ~coord_df["inside_rough_bandung_raya_box"]).sum())},
    {"metric": "coordinate_verified_true", "count": int(coord_df["coordinate_verified_bool"].sum())},
    {"metric": "active_coordinate_unverified", "count": int((coord_df["display_status"].eq("active_candidate") & ~coord_df["coordinate_verified_bool"]).sum())},
    {"metric": "distance_available", "count": int(coord_df["distance_num"].notna().sum())},
])
display(coord_summary)

coord_attention = coord_df[
    coord_df["display_status"].eq("active_candidate") &
    ((~coord_df["coordinate_verified_bool"]) | (~coord_df["coord_numeric_ok"]) | (~coord_df["inside_rough_bandung_raya_box"]))
].copy()
attention_cols = [c for c in ["location_id", "location_name", "latitude", "longitude", "coordinate_verified", "distance_from_alun_alun_km", "coordinate_audit_reason"] if c in coord_attention.columns]
display(coord_attention[attention_cols].sort_values("location_name").head(30))

coordinate_audit_path = CURATED_DIR / "coordinate_audit.csv"
if coordinate_audit_path.exists():
    coordinate_audit_df = pd.read_csv(coordinate_audit_path, dtype=str, keep_default_na=False)
    display(coordinate_audit_df.head(20))
else:
    print("coordinate_audit.csv tidak ditemukan")


,metric,count
0,total_rows,234
1,numeric_coordinate_ok,234
2,missing_or_invalid_coordinate,0
3,inside_rough_bandung_raya_box,233
4,outside_rough_bandung_raya_box,1
5,coordinate_verified_true,220
6,active_coordinate_unverified,9
7,distance_available,234


,location_id,location_name,latitude,longitude,coordinate_verified,distance_from_alun_alun_km,coordinate_audit_reason
224,LOC-225,Curug Ceret Pangalengan,-7.3620716,107.5468974,True,49.41,
22,LOC-023,Curug Malela,-7.0115741,107.2056616,False,45.42,outside broad Bandung Raya coordinate bounds
154,LOC-155,Curug Panganten,-6.8435086,107.5639385,False,9.92,Pangalengan/Malabar coordinate too close to Bandung center: 9.9 km from Alun-Alun Bandung
43,LOC-044,Jans Park (Jatinangor National Park),-6.918795,107.769342,False,17.92,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 17.9...
137,LOC-138,Kebun Teh Sukawana,-6.89556,107.58361,False,3.9,Sukawana/Lembang coordinate too close to Bandung center: 3.9 km from Alun-Alun Bandung
165,LOC-166,Muara Rahong Hills,-6.9325,107.5625,False,5.06,Pangalengan/Malabar coordinate too close to Bandung center: 5.1 km from Alun-Alun Bandung
68,LOC-069,Pemandian Cipanas Cileungsing,-7.00583,107.51944,False,13.45,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 13.5...
161,LOC-162,Rumah Putih Cukul,-7.0315,107.7203,False,17.47,Pangalengan/Malabar coordinate too close to Bandung center: 17.5 km from Alun-Alun Ban...
157,LOC-158,Situ Ninah (Situ Datar),-7.03278,107.56611,False,13.15,Pangalengan/Malabar coordinate too close to Bandung center: 13.1 km from Alun-Alun Ban...
115,LOC-116,Wisata Kampoeng Ciherang,-6.8293636,107.7978925,False,23.44,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 23.4...


,location_name,category,display_status,curation_action,latitude,longitude,coordinate_verified,distance_from_alun_alun_km,coordinate_audit_reason
0,Curug Malela,Wisata Alam,active_candidate,keep,-7.0115741,107.2056616,False,45.42,outside broad Bandung Raya coordinate bounds
1,Gn. Tampomas,Rekreasi Keluarga,exclude_scope,remove,-6.8817995,107.5683276,False,6.17,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 6.2 ...
2,Jans Park (Jatinangor National Park),Rekreasi Keluarga,active_candidate,keep,-6.918795,107.769342,False,17.92,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 17.9...
3,Masjid Agung Sumedang,Tempat Ibadah,exclude_scope,remove,-6.907497,107.647137,False,4.7,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 4.7 ...
4,Pemandian Cipanas Cileungsing,Wisata Alam,active_candidate,keep,-7.00583,107.51944,False,13.45,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 13.5...
5,Wisata Kampoeng Ciherang,Rekreasi Keluarga,active_candidate,keep,-6.8293636,107.7978925,False,23.44,Sumedang/Jatigede coordinate too close to Bandung center or outside strict scope: 23.4...
6,Kebun Teh Sukawana,Wisata Alam,active_candidate,keep,-6.89556,107.58361,False,3.9,Sukawana/Lembang coordinate too close to Bandung center: 3.9 km from Alun-Alun Bandung
7,Curug Panganten,Wisata Alam,active_candidate,keep,-6.8435086,107.5639385,False,9.92,Pangalengan/Malabar coordinate too close to Bandung center: 9.9 km from Alun-Alun Bandung
8,Situ Ninah (Situ Datar),Wisata Alam,active_candidate,keep,-7.03278,107.56611,False,13.15,Pangalengan/Malabar coordinate too close to Bandung center: 13.1 km from Alun-Alun Ban...
9,Rumah Putih Cukul,Rekreasi Keluarga,active_candidate,keep,-7.0315,107.7203,False,17.47,Pangalengan/Malabar coordinate too close to Bandung center: 17.5 km from Alun-Alun Ban...


### Keputusan: fitur jarak bisa dipakai dengan guardrail

Koordinat sudah cukup untuk fitur radius dan jarak, tetapi baris yang belum `coordinate_verified=True` tetap harus diberi perlakuan hati-hati. Untuk lokasi yang masuk daftar perhatian, sistem boleh menghitung jarak, tetapi tidak boleh terlalu percaya diri pada klaim presisi.


## Fase F - Harga dan Jam Buka

Tujuan: mengecek apakah harga dan jam operasional cukup rapi untuk filter gratis/berbayar, range harga, buka sekarang, weekday, dan weekend.


In [7]:
price_time_cols = [
    "location_id", "location_name", "display_status", "price_min", "price_max", "price_type", "price_verified",
    "jam_buka", "jam_tutup", "jam_buka_weekday", "jam_tutup_weekday", "jam_buka_weekend", "jam_tutup_weekend",
    "status_jam", "sumber_jam", "open_24h_verified", "night_verified", "catatan_jam",
]
price_time_cols = [c for c in price_time_cols if c in df.columns]
pt = df[price_time_cols].copy()

for col in ["price_min", "price_max"]:
    if col in pt.columns:
        pt[col + "_num"] = pd.to_numeric(pt[col].astype(str).str.replace(r"[^0-9.-]", "", regex=True), errors="coerce")

price_summary = pd.DataFrame([
    {"metric": "price_min_available", "count": int(pt.get("price_min_num", pd.Series(dtype=float)).notna().sum())},
    {"metric": "price_max_available", "count": int(pt.get("price_max_num", pd.Series(dtype=float)).notna().sum())},
    {"metric": "active_price_verified_true", "count": int((pt["display_status"].eq("active_candidate") & pt.get("price_verified", "").astype(str).str.lower().eq("true")).sum()) if "price_verified" in pt.columns else None},
    {"metric": "active_price_verified_not_true", "count": int((pt["display_status"].eq("active_candidate") & ~pt.get("price_verified", "").astype(str).str.lower().eq("true")).sum()) if "price_verified" in pt.columns else None},
])
display(price_summary)

if "price_type" in pt.columns:
    price_type_counts = pt.groupby(["display_status", "price_type"], dropna=False).size().reset_index(name="count").sort_values(["display_status", "count"], ascending=[True, False])
    display(price_type_counts)

zero_price_issue = pt[
    pt.get("price_max_num", pd.Series(index=pt.index, dtype=float)).eq(0) &
    ~pt.get("price_type", pd.Series(index=pt.index, dtype=str)).astype(str).str.lower().str.contains("gratis|free", na=False)
].copy()
zero_cols = [c for c in ["location_id", "location_name", "price_min", "price_max", "price_type", "price_verified"] if c in zero_price_issue.columns]
display(zero_price_issue[zero_cols].head(20))

hour_cols = ["jam_buka_weekday", "jam_tutup_weekday", "jam_buka_weekend", "jam_tutup_weekend"]
for col in hour_cols:
    if col not in pt.columns:
        pt[col] = ""

pt["weekday_hours_complete"] = pt["jam_buka_weekday"].astype(str).str.strip().ne("") & pt["jam_tutup_weekday"].astype(str).str.strip().ne("")
pt["weekend_hours_complete"] = pt["jam_buka_weekend"].astype(str).str.strip().ne("") & pt["jam_tutup_weekend"].astype(str).str.strip().ne("")
pt["open_24h_bool"] = pt.get("open_24h_verified", "").astype(str).str.lower().eq("true") if "open_24h_verified" in pt.columns else False
pt["night_bool"] = pt.get("night_verified", "").astype(str).str.lower().eq("true") if "night_verified" in pt.columns else False

hour_summary = pd.DataFrame([
    {"metric": "weekday_hours_complete", "count": int(pt["weekday_hours_complete"].sum())},
    {"metric": "weekend_hours_complete", "count": int(pt["weekend_hours_complete"].sum())},
    {"metric": "active_weekday_hours_missing", "count": int((pt["display_status"].eq("active_candidate") & ~pt["weekday_hours_complete"]).sum())},
    {"metric": "active_weekend_hours_missing", "count": int((pt["display_status"].eq("active_candidate") & ~pt["weekend_hours_complete"]).sum())},
    {"metric": "open_24h_verified_true", "count": int(pt["open_24h_bool"].sum())},
    {"metric": "night_verified_true", "count": int(pt["night_bool"].sum())},
])
display(hour_summary)

hour_attention = pt[
    pt["display_status"].eq("active_candidate") &
    ((~pt["weekday_hours_complete"]) | (~pt["weekend_hours_complete"]) | pt["open_24h_bool"])
].copy()
hour_attention_cols = [c for c in ["location_id", "location_name", "jam_buka_weekday", "jam_tutup_weekday", "jam_buka_weekend", "jam_tutup_weekend", "open_24h_verified", "night_verified", "status_jam", "catatan_jam"] if c in hour_attention.columns]
display(hour_attention[hour_attention_cols].head(30))


,metric,count
0,price_min_available,234
1,price_max_available,234
2,active_price_verified_true,209
3,active_price_verified_not_true,0


,display_status,price_type,count
2,active_candidate,Per Orang,157
1,active_candidate,Gratis,51
0,active_candidate,Berbayar,1
5,exclude_scope,Per Orang,14
4,exclude_scope,Gratis,4
3,exclude_scope,Berbayar,1
6,status_uncertain,Per Orang,3
7,temporarily_hidden,Per Orang,3


,location_id,location_name,price_min,price_max,price_type,price_verified
232,LOC-233,Kampung Wisata Pangjugjugan,0,0,Berbayar,False


,metric,count
0,weekday_hours_complete,233
1,weekend_hours_complete,233
2,active_weekday_hours_missing,0
3,active_weekend_hours_missing,0
4,open_24h_verified_true,15
5,night_verified_true,36


,location_id,location_name,jam_buka_weekday,jam_tutup_weekday,jam_buka_weekend,jam_tutup_weekend,open_24h_verified,night_verified,status_jam,catatan_jam
11,LOC-012,Bukit Moko Bandung,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
41,LOC-042,Gunung Putri Lembang,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
67,LOC-068,Pasar Cimindi,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
85,LOC-086,Taman Alun-Alun Kota Cimahi,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
88,LOC-089,Taman Film,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
91,LOC-092,Taman Jomblo,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
95,LOC-096,Taman Lansia,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
97,LOC-098,Taman Musik,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
98,LOC-099,Taman Superhero,00:00,23:59,00:00,23:59,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.
133,LOC-134,Bukit Bintang Bandung (Patahan Lembang),06:00,17:00,06:00,17:00,True,True,general_schedule_used_for_both,Jadwal umum digunakan untuk weekday dan weekend.


### Keputusan: filter harga dan jam buka bisa dipakai, tetapi tidak boleh dianggap sempurna

Kolom harga dan jam operasional sudah tersedia untuk filter dasar. Baris dengan harga nol yang tidak konsisten, jam kosong, atau `open_24h_verified` yang perlu dicek tetap masuk daftar perhatian agar filter ?gratis? dan ?buka sekarang? tidak memberi klaim keliru.


## Ringkasan Sementara Setelah Fase D-F

| Area | Keputusan |
|---|---|
| Scope destinasi | Rekomendasi fokus ke `active_candidate` |
| Status non-aktif | Ditahan dari ranking utama, bukan dihapus |
| Koordinat | Cukup untuk distance/radius dengan caveat pada unverified rows |
| Harga | Bisa dipakai untuk filter dasar, tetap perlu guardrail pada kasus kosong/aneh |
| Jam buka | Bisa dipakai untuk filter dasar, tetapi baris bermasalah tetap perlu dicek |
| Tahap berikutnya | Fase G-I: fasilitas, media, dan sentiment lineage |

Tahap ini masih penyiapan data wisata. Belum masuk ke LLM dan belum mengubah ranking utama.
